In [16]:
import dashscope
import os 
from dotenv import load_dotenv
from dashscope import Generation
import time
import random
import re
from typing import List, Dict, Optional, Any, Callable
from functools import wraps
load_dotenv()
print(os.getenv('DASHSCOPE_API_KEY'))
print("环境导入完毕")




sk-6f8168916cc9407c90c37ad4a257e0e0
环境导入完毕


In [15]:
# ============================================================
# 任务1：通义千问API调用基础
# ============================================================

"""
任务：实现一个简单的聊天补全函数
要求：
1. 使用通义千问API（dashscope库）
2. 支持模型选择（qwen-turbo / qwen-plus / qwen-max）
3. 返回响应内容和Token使用量

验收标准：
✅ 函数能正常调用API并返回结果
✅ 能选择不同的模型
✅ 返回包含content和usage的字典
"""
def chanllenge1(text: str,systen_message:str = "你是一个接话者,你能接住任何话题并延伸", model: str = "qwen-turbo"):
    response = Generation.call(
        model=model,
        messages=[{"role":"system","content":systen_message},{"role": "user", "content": text}],
        temperature=1.0,
        max_tokens=300,
        result_format="message"
        
    )
    print(response)
    if response.status_code == 200:
        return {
            "content": response.output.choices[0].message.content,
            "usage": {
                "prompt_tokens":response.usage.input_tokens,
                 "response_tokens":response.usage.output_tokens
                 }
            }
    else:
        return {"error": f"{response.code}-{response.message}"} 
print(chanllenge1(text="我跳起来就") )

{"status_code": 200, "request_id": "094a8f6c-10a5-921c-be84-6649a5a0c575", "code": "", "message": "", "output": {"text": null, "finish_reason": null, "choices": [{"finish_reason": "stop", "message": {"role": "assistant", "content": "我跳起来就看到一只大黄狗朝我狂奔过来，吓得我差点没站稳。还好它只是在追自己的尾巴，转了个圈又跑回去了。你遇到过这种有趣的事吗？有时候生活就像这只狗，看似疯狂，其实只是在找点乐子。"}}]}, "usage": {"input_tokens": 34, "output_tokens": 63, "prompt_tokens_details": {"cached_tokens": 0}, "total_tokens": 97}}
{'content': '我跳起来就看到一只大黄狗朝我狂奔过来，吓得我差点没站稳。还好它只是在追自己的尾巴，转了个圈又跑回去了。你遇到过这种有趣的事吗？有时候生活就像这只狗，看似疯狂，其实只是在找点乐子。', 'usage': {'prompt_tokens': 34, 'response_tokens': 63}}


In [17]:
# ============================================================
# 任务2：多模型管理器
# ============================================================

"""
任务：实现多模型管理器
要求：
1. 支持通义千问主模型（qwen-turbo / qwen-plus / qwen-max）
2. 支持文心一言、智谱GLM作为备选
3. 统一调用接口

验收标准：
✅ 支持切换不同模型
✅ 统一返回格式
✅ 成都教育科技场景示例
"""
class ModelManager:
    """
    多模型管理器（通义千问为主）
    
    支持统一接口调用不同的AI模型，自动处理模型差异
    
    iOS类比：设计模式中的 Adapter Pattern，统一不同接口
    """
    
    def __init__(self):
        """初始化管理器，设置默认模型"""
        self.default_model = "qwen-turbo"
        
        # 模型配置表（人民币计价）
        self.model_config = {
            # 通义千问系列（主用）
            "qwen-turbo": {
                "provider": "aliyun",
                "context_window": 8000,
                "cost_per_1k_input": 0.004,  # ¥
                "cost_per_1k_output": 0.012,  # ¥
                "best_for": ["日常对话", "轻量任务", "快速响应"]
            },
            "qwen-plus": {
                "provider": "aliyun", 
                "context_window": 128000,
                "cost_per_1k_input": 0.004,  # ¥
                "cost_per_1k_output": 0.012,  # ¥
                "best_for": ["复杂推理", "代码生成", "企业应用"]
            },
            "qwen-max": {
                "provider": "aliyun",
                "context_window": 32000,
                "cost_per_1k_input": 0.04,  # ¥
                "cost_per_1k_output": 0.12,  # ¥
                "best_for": ["最高质量", "专业任务", "深度分析"]
            },
            # 文心一言（备选）
            "ernie-bot-4": {
                "provider": "baidu",
                "context_window": 8000,
                "cost_per_1k_input": 0.02,  # ¥
                "cost_per_1k_output": 0.06,  # ¥
                "best_for": ["中文对话", "百度生态", "国内合规"]
            },
            # 智谱GLM（备选）
            "glm-4": {
                "provider": "zhipuai",
                "context_window": 128000,
                "cost_per_1k_input": 0.01,  # ¥
                "cost_per_1k_output": 0.01,  # ¥
                "best_for": ["学术研究", "中英双语", "代码任务"]
            },
        }
    
    def chat(
        self,
        message: str,
        model: Optional[str] = None,
        **kwargs
    ) -> Dict[str, Any]:
        """
        统一聊天接口
        
        Args:
            message: 用户消息
            model: 模型名称，默认使用 self.default_model
            **kwargs: 传递给具体模型的参数
            
        Returns:
            统一格式的响应字典
        """
        model = model or self.default_model
        
        if model not in self.model_config:
            return {
                "error": f"未知模型: {model}",
                "content": None,
                "usage": None
            }
        
        config = self.model_config[model]
        
        # 根据provider调用不同的API
        if config["provider"] == "aliyun":
            return self._chat_qwen(message, model, **kwargs)
        elif config["provider"] == "baidu":
            return self._chat_ernie(message, model, **kwargs)
        elif config["provider"] == "zhipuai":
            return self._chat_glm(message, model, **kwargs)
        else:
            return {"error": "不支持的provider", "content": None, "usage": None}
    
    def _chat_qwen(
        self, 
        message: str, 
        model: str, 
        **kwargs
    ) -> Dict[str, Any]:
        """调用通义千问API"""
        import dashscope
        from dashscope import Generation
        
        dashscope.api_key = os.getenv("DASHSCOPE_API_KEY")
        
        try:
            response = Generation.call(
                model=model,
                messages=[{"role": "user", "content": message}],
                temperature=kwargs.get("temperature", 0.7),
                max_tokens=kwargs.get("max_tokens", 1000),
                result_format='message',
            )
            
            if response.status_code == 200:
                return {
                    "content": response.output.choices[0].message.content,
                    "usage": {
                        "prompt_tokens": response.usage.input_tokens,
                        "completion_tokens": response.usage.output_tokens,
                        "total_tokens": response.usage.total_tokens,
                    },
                    "model": model,
                    "cost_estimate": self.estimate_cost(
                        response.usage.input_tokens,
                        response.usage.output_tokens,
                        model
                    )
                }
            else:
                return {"error": f"{response.code} - {response.message}", "content": None, "usage": None}
        except Exception as e:
            return {"error": str(e), "content": None, "usage": None}
    
    def _chat_ernie(
        self, 
        message: str, 
        model: str, 
        **kwargs
    ) -> Dict[str, Any]:
        """调用文心一言API（需要安装erniebot库）"""
        # 实际使用时需要安装：pip install erniebot
        try:
            import erniebot
            
            erniebot.api_key = os.getenv("ERNIE_API_KEY")
            
            response = erniebot.ChatCompletion.create(
                model=model,
                messages=[{"role": "user", "content": message}],
            )
            
            return {
                "content": response.result,
                "usage": {
                    "prompt_tokens": response.usage.prompt_tokens,
                    "completion_tokens": response.usage.completion_tokens,
                    "total_tokens": response.usage.total_tokens,
                },
                "model": model,
                "cost_estimate": self.estimate_cost(
                    response.usage.prompt_tokens,
                    response.usage.completion_tokens,
                    model
                )
            }
        except ImportError:
            return {"error": "请安装erniebot库: pip install erniebot", "content": None, "usage": None}
        except Exception as e:
            return {"error": str(e), "content": None, "usage": None}
    
    def _chat_glm(
        self, 
        message: str, 
        model: str, 
        **kwargs
    ) -> Dict[str, Any]:
        """调用智谱GLM API（需要安装zhipuai库）"""
        # 实际使用时需要安装：pip install zhipuai
        try:
            import zhipuai
            
            zhipuai.api_key = os.getenv("ZHIPU_API_KEY")
            
            response = zhipuai.model_api.invoke(
                model=model,
                prompt=[{"role": "user", "content": message}],
            )
            
            if response.get("code") == 200:
                return {
                    "content": response["data"]["choices"][0]["content"],
                    "usage": {"total_tokens": len(message) // 4},
                    "model": model,
                    "cost_estimate": self.estimate_cost(len(message) // 4, 50, model)
                }
            else:
                return {"error": response.get("msg", "Unknown error"), "content": None, "usage": None}
        except ImportError:
            return {"error": "请安装zhipuai库: pip install zhipuai", "content": None, "usage": None}
        except Exception as e:
            return {"error": str(e), "content": None, "usage": None}
    
    def estimate_cost(
        self, 
        prompt_tokens: int, 
        completion_tokens: int, 
        model: str
    ) -> float:
        """
        预估API调用成本
        
        Args:
            prompt_tokens: 输入Token数
            completion_tokens: 输出Token数
            model: 模型名称
            
        Returns:
            预估成本（人民币元）
        """
        if model not in self.model_config:
            return 0.0
        
        config = self.model_config[model]
        
        input_cost = (prompt_tokens / 1000) * config["cost_per_1k_input"]
        output_cost = (completion_tokens / 1000) * config["cost_per_1k_output"]
        
        return input_cost + output_cost
    
    def get_model_info(self, model: str) -> Dict[str, Any]:
        """获取模型信息"""
        return self.model_config.get(model, {})
    
    def list_models(self) -> List[str]:
        """列出所有可用模型"""
        return list(self.model_config.keys())

In [ ]:
"""
任务：实现Token计数器
要求：
1. 支持中文和英文文本的Token估算
2. 支持消息列表Token计算
3. 计算API调用的成本

验收标准：
✅ 能计算单段文本的Token数（估算）
✅ 能计算消息列表的总Token数
✅ 能预估API调用的成本
"""

def count_tokens_cn(text: str) -> int:
    """
    估算中文文本的Token数
    
    经验估算规则：
    - 中文：约1.5个字符 ≈ 1 token
    - 英文：约4个字符 ≈ 1 token
    - 标点/数字：约2个 ≈ 1 token
    
    Args:
        text: 待计算的文本
        
    Returns:
        估算的Token数量
    """
    # 分离中英文
    chinese_chars = len(re.findall(r'[\u4e00-\u9fff]', text))
    english_chars = len(re.findall(r'[a-zA-Z]', text))
    other_chars = len(text) - chinese_chars - english_chars
    
    # 估算Token
    estimated_tokens = chinese_chars / 1.5 + english_chars / 4 + other_chars / 2
    
    return int(estimated_tokens)

class TokenCounter:
    """
    Token计数器
    
    支持中文和英文文本的Token估算
    
    Reference: https://help.aliyun.com/zh/model-studio/developer-reference/use-qwen-by-calling-api
    """
    
    def __init__(self, model: str = "qwen-turbo"):
        """
        初始化计数器
        
        Args:
            model: 模型名称（影响上下文窗口参考）
        """
        self.model = model
    
    def count_text(self, text: str) -> int:
        """
        计算单段文本的Token数
        
        Args:
            text: 待计算的文本
            
        Returns:
            Token数量（估算）
        """
        return count_tokens_cn(text)
    
    def count_messages(
        self, 
        messages: List[Dict[str, str]]
    ) -> int:
        """
        计算消息列表的总Token数
        
        Args:
            messages: 消息列表，格式为 [{"role": "system"/"user"/"assistant", "content": "..."}]
            
        Returns:
            总Token数（估算）
        """
        # 每个消息的基础开销
        tokens_per_message = 4
        
        total_tokens = 0
        
        for message in messages:
            # 角色Token（估算）
            total_tokens += 4
            
            # 内容Token
            total_tokens += count_tokens_cn(message.get("content", ""))
            
            # 名字Token（如果有）
            if "name" in message:
                total_tokens += count_tokens_cn(message.get("name", ""))
                total_tokens += 1  # name字段额外开销
            
            # 消息基础开销
            total_tokens += tokens_per_message
        
        # 每个消息列表的额外开销
        total_tokens += 3
        
        return total_tokens
    
    def calculate_cost(
        self,
        prompt_tokens: int,
        completion_tokens: int,
        model: str = "qwen-turbo"
    ) -> Dict[str, float]:
        """
        计算API调用成本（人民币元）
        
        Args:
            prompt_tokens: 输入Token数
            completion_tokens: 输出Token数
            model: 模型名称
            
        Returns:
            成本明细字典
        """
        # 价格表（¥/1M Tokens）
        prices = {
            "qwen-turbo": {"input": 4.0, "output": 12.0},
            "qwen-plus": {"input": 4.0, "output": 12.0},
            "qwen-max": {"input": 40.0, "output": 120.0},
            "ernie-bot-4": {"input": 20.0, "output": 60.0},
            "glm-4": {"input": 10.0, "output": 10.0},
        }
        
        model_prices = prices.get(model, prices["qwen-turbo"])
        
        input_cost = (prompt_tokens / 1_000_000) * model_prices["input"]
        output_cost = (completion_tokens / 1_000_000) * model_prices["output"]
        
        return {
            "input_cost": input_cost,
            "output_cost": output_cost,
            "total_cost": input_cost + output_cost,
        }
    
    def truncate_messages(
        self,
        messages: List[Dict[str, str]],
        max_tokens: int,
        preserve_system: bool = True
    ) -> List[Dict[str, str]]:
        """
        智能截断消息列表
        
        当消息过长时，保留最近的消息直到不超过max_tokens
        
        Args:
            messages: 原始消息列表
            max_tokens: 最大Token数
            preserve_system: 是否保留系统消息
            
        Returns:
            截断后的消息列表
        """
        # 分离系统消息和其他消息
        system_message = None
        conversation = []
        
        if preserve_system and messages and messages[0]["role"] == "system":
            system_message = messages[0]
            conversation = messages[1:]
        else:
            conversation = messages
        
        result = []
        total_tokens = 0
        
        # 从最新消息开始添加
        for message in reversed(conversation):
            msg_tokens = count_tokens_cn(message.get("content", ""))
            msg_tokens += 4  # 消息基础开销
            
            if total_tokens + msg_tokens <= max_tokens:
                result.insert(0, message)
                total_tokens += msg_tokens
            else:
                break
        
        # 重新添加系统消息
        if system_message:
            system_tokens = count_tokens_cn(system_message["content"]) + 4
            if total_tokens + system_tokens <= max_tokens:
                result.insert(0, system_message)
        
        return result

In [ ]:
# ============================================================
# 任务4：错误处理与重试
# ============================================================

"""
任务：实现带重试的API调用
要求：
1. 处理RateLimitError、RequestTimeout等错误
2. 指数退避重试
3. 最大重试3次

验收标准：
✅ 装饰器/函数能自动重试失败的请求
✅ 使用指数退避策略
✅ 区分可重试和不可重试的错误
"""

class RetryConfig:
    """重试配置"""
    
    def __init__(
        self,
        max_attempts: int = 3,
        base_delay: float = 1.0,
        max_delay: float = 60.0,
        jitter: bool = True
    ):
        """
        初始化重试配置
        
        Args:
            max_attempts: 最大尝试次数
            base_delay: 基础延迟时间（秒）
            max_delay: 最大延迟时间（秒）
            jitter: 是否添加随机抖动
        """
        self.max_attempts = max_attempts
        self.base_delay = base_delay
        self.max_delay = max_delay
        self.jitter = jitter


def with_retry(config: Optional[RetryConfig] = None):
    """
    重试装饰器
    
    使用指数退避策略处理可重试的错误
    
    Args:
        config: RetryConfig配置对象
        
    Reference:
        https://help.aliyun.com/zh/model-studio/developer-reference/use-qwen-by-calling-api
    """
    if config is None:
        config = RetryConfig()
    
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs) -> Any:
            last_exception = None
            
            for attempt in range(config.max_attempts):
                try:
                    # 尝试执行函数
                    return func(*args, **kwargs)
                    
                except Exception as e:
                    last_exception = e
                    error_type = type(e).__name__
                    error_str = str(e).lower()
                    
                    # 判断是否应该重试
                    should_retry = False
                    retry_message = ""
                    
                    # Rate Limit 错误
                    if "rate" in error_str or "429" in error_str or "throttling" in error_str:
                        should_retry = True
                        retry_message = "⏳ Rate Limit触发"
                    
                    # 超时错误
                    elif "timeout" in error_str or "timeout" in error_type:
                        should_retry = True
                        retry_message = "⏱️ 请求超时"
                    
                    # 服务器错误（5xx）
                    elif "500" in error_str or "503" in error_str or "internal" in error_str:
                        should_retry = True
                        retry_message = f"🔧 服务器错误"
                    
                    # 服务不可用
                    elif "unavailable" in error_str or "unavailable" in error_type:
                        should_retry = True
                        retry_message = "⚠️ 服务不可用"
                    
                    # 不可重试的错误
                    if not should_retry:
                        print(f"❌ 不可重试的错误: {error_type}")
                        raise
                    
                    # 检查是否还有重试机会
                    if attempt >= config.max_attempts - 1:
                        print(f"❌ 达到最大重试次数 ({config.max_attempts})")
                        raise
                    
                    # 计算延迟时间（指数退避）
                    delay = min(
                        config.base_delay * (2 ** attempt),
                        config.max_delay
                    )
                    
                    # 添加随机抖动，避免多请求同时重试
                    if config.jitter:
                        delay *= random.uniform(1.0, 1.5)
                    
                    print(f"{retry_message}，{delay:.1f}s后重试 ({attempt + 1}/{config.max_attempts})...")
                    time.sleep(delay)
            
            # 所有重试都失败
            raise last_exception
        
        return wrapper
    return decorator


class APIClientWithRetry:
    """
    带重试机制的API客户端（通义千问版）
    
    iOS类比：封装了网络请求重试逻辑的NetworkManager
    """
    
    def __init__(self, retry_config: Optional[RetryConfig] = None):
        self.retry_config = retry_config or RetryConfig()
        self.client = None
    
    def _get_client(self):
        """延迟初始化客户端"""
        if self.client is None:
            import dashscope
            from dashscope import Generation
            dashscope.api_key = os.getenv("DASHSCOPE_API_KEY")
            self.client = Generation
        return self.client
    
    def chat(
        self,
        messages: List[Dict[str, str]],
        model: str = "qwen-turbo",
        **kwargs
    ) -> Dict[str, Any]:
        """
        带重试的聊天请求
        
        Args:
            messages: 消息列表
            model: 模型名称
            **kwargs: 其他参数
            
        Returns:
            API响应字典
        """
        @with_retry(self.retry_config)
        def _request():
            client = self._get_client()
            response = client.call(
                model=model,
                messages=messages,
                result_format='message',
                **kwargs
            )
            
            if response.status_code != 200:
                raise Exception(f"{response.code} - {response.message}")
            
            return response
        
        response = _request()
        
        return {
            "content": response.output.choices[0].message.content,
            "usage": {
                "prompt_tokens": response.usage.input_tokens,
                "completion_tokens": response.usage.output_tokens,
                "total_tokens": response.usage.total_tokens,
            },
            "model": model,
        }